# ZGLMM159 — M1 — Planejamento MRP e depósito — Cenário Centro 4058

**View:** `dev_procurement.corp_curated.vw_zglmm159_m1`  
**Transação SAP:** ZGLMM159  
**Cenário:** `cod_centro = '4058'`  
**Objetivo:** repetir no novo desenho C1/M1/M2 o mesmo cenário validado anteriormente para a ZGLMM159, preservando a granularidade específica de cada visão.  
**Colunas no DESCRIBE recebido:** 26

## Como usar
Execute **Run All** no Databricks. As consultas leem a view diretamente e não criam tabelas ou views temporárias.

## Atenções iniciais
- A chave abaixo é apenas uma **candidata** e será confirmada pelos testes de unicidade.
- Colunas comentadas como sem fonte ou `NULL` serão medidas, não presumidas.
- Volumetria igual não comprova qualidade dos dados.

**Referência histórica do cenário:** centro 4058. Na validação anterior, C1 e M2 apresentaram granularidade de uma linha por material, enquanto M1 apresentou granularidade de material + depósito. Os testes abaixo reconfirmam esse comportamento nas novas views.


## 1. Metadados e definição da view


In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.vw_zglmm159_m1;


In [0]:
SHOW CREATE TABLE dev_procurement.corp_curated.vw_zglmm159_m1;


## Preparação do cenário — centro 4058

A view temporária abaixo garante que todas as análises usem exatamente o mesmo recorte. A contagem de controle confirma que nenhum outro centro entrou no cenário.


In [0]:
CREATE OR REPLACE TEMP VIEW tmp_zglmm159_m1_centro_4058 AS
SELECT *
FROM dev_procurement.corp_curated.vw_zglmm159_m1
WHERE TRIM(CAST(cod_centro AS STRING)) = '4058';


In [0]:
SELECT
  COUNT(*) AS total_linhas_cenario,
  COUNT(DISTINCT cod_centro) AS centros_distintos,
  MIN(cod_centro) AS centro_minimo,
  MAX(cod_centro) AS centro_maximo
FROM tmp_zglmm159_m1_centro_4058;


## 2. Volumetria geral


In [0]:
SELECT
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT cod_material) AS materiais_distintos,
  COUNT(DISTINCT cod_centro) AS centros_distintos,
  COUNT(DISTINCT cod_empresa) AS empresas_distintas
  , COUNT(DISTINCT cod_deposito) AS depositos_distintos
FROM tmp_zglmm159_m1_centro_4058;


## 3. Distribuição por centro e empresa — base para definição de cenários


In [0]:
SELECT
  cod_centro,
  cod_empresa,
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_material) AS materiais,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_linhas
FROM tmp_zglmm159_m1_centro_4058
GROUP BY cod_centro, cod_empresa
ORDER BY linhas DESC, cod_centro, cod_empresa;


In [0]:
SELECT
  cod_centro,
  cod_deposito,
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_material) AS materiais,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_linhas
FROM tmp_zglmm159_m1_centro_4058
GROUP BY cod_centro, cod_deposito
ORDER BY linhas DESC, cod_centro, cod_deposito;


## 4. Granularidade e testes de chave candidata


In [0]:
SELECT 'cod_material' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_deposito` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_deposito` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_material + cod_centro + cod_deposito + cod_empresa' AS chave, COUNT(*) AS linhas, COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_deposito` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))) AS distintos,
ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_deposito` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))), 0), 6) AS linhas_por_chave
FROM tmp_zglmm159_m1_centro_4058;


### Duplicidades na chave candidata: `cod_material + cod_centro + cod_deposito + cod_empresa`


In [0]:
SELECT `cod_material`, `cod_centro`, `cod_deposito`, `cod_empresa`, COUNT(*) AS qtd
FROM tmp_zglmm159_m1_centro_4058
GROUP BY `cod_material`, `cod_centro`, `cod_deposito`, `cod_empresa`
HAVING COUNT(*) > 1
ORDER BY qtd DESC, `cod_material`, `cod_centro`, `cod_deposito`, `cod_empresa`;


### Inspeção campo a campo das chaves repetidas


In [0]:
WITH duplicadas AS (
  SELECT `cod_material`, `cod_centro`, `cod_deposito`, `cod_empresa`
  FROM tmp_zglmm159_m1_centro_4058
  GROUP BY `cod_material`, `cod_centro`, `cod_deposito`, `cod_empresa`
  HAVING COUNT(*) > 1
)
SELECT t.*
FROM tmp_zglmm159_m1_centro_4058 t
INNER JOIN duplicadas d ON COALESCE(CAST(t.`cod_material` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_material` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_centro` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_centro` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_deposito` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_deposito` AS STRING), '<NULL>') AND COALESCE(CAST(t.`cod_empresa` AS STRING), '<NULL>') = COALESCE(CAST(d.`cod_empresa` AS STRING), '<NULL>')
ORDER BY t.`cod_material`, t.`cod_centro`, t.`cod_deposito`, t.`cod_empresa`;


## 5. Preenchimento de todas as colunas — nulo, zero e útil


In [0]:
SELECT 'cod_material' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_material` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_material` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_material` IS NULL OR LOWER(TRIM(CAST(`cod_material` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_deposito' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_deposito` IS NULL OR LOWER(TRIM(CAST(`cod_deposito` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_deposito` IS NULL OR LOWER(TRIM(CAST(`cod_deposito` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_deposito` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_deposito` IS NULL OR LOWER(TRIM(CAST(`cod_deposito` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_deposito` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_deposito` IS NULL OR LOWER(TRIM(CAST(`cod_deposito` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_deposito` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_centro' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_centro` IS NULL OR LOWER(TRIM(CAST(`cod_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_empresa' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_empresa` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_empresa` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_empresa` IS NULL OR LOWER(TRIM(CAST(`cod_empresa` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'tp_grupo_mrp' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `tp_grupo_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`tp_grupo_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`tp_grupo_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`tp_grupo_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`tp_grupo_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `tp_grupo_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`tp_grupo_mrp` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_regra_calculo_tamanho_lote' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_regra_calculo_tamanho_lote` IS NULL OR LOWER(TRIM(CAST(`cod_regra_calculo_tamanho_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_regra_calculo_tamanho_lote` IS NULL OR LOWER(TRIM(CAST(`cod_regra_calculo_tamanho_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_regra_calculo_tamanho_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_regra_calculo_tamanho_lote` IS NULL OR LOWER(TRIM(CAST(`cod_regra_calculo_tamanho_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_regra_calculo_tamanho_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_regra_calculo_tamanho_lote` IS NULL OR LOWER(TRIM(CAST(`cod_regra_calculo_tamanho_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_regra_calculo_tamanho_lote` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'tp_mrp' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `tp_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`tp_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`tp_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`tp_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`tp_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `tp_mrp` IS NULL OR LOWER(TRIM(CAST(`tp_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`tp_mrp` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_planejador_mrp' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_planejador_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_planejador_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_planejador_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_planejador_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_planejador_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_planejador_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_planejador_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_planejador_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_planejador_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_planejador_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_planejador_mrp` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_horizonte_planejamento_fixo' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_horizonte_planejamento_fixo` IS NULL OR LOWER(TRIM(CAST(`cod_horizonte_planejamento_fixo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_horizonte_planejamento_fixo` IS NULL OR LOWER(TRIM(CAST(`cod_horizonte_planejamento_fixo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_horizonte_planejamento_fixo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_horizonte_planejamento_fixo` IS NULL OR LOWER(TRIM(CAST(`cod_horizonte_planejamento_fixo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_horizonte_planejamento_fixo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_horizonte_planejamento_fixo` IS NULL OR LOWER(TRIM(CAST(`cod_horizonte_planejamento_fixo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_horizonte_planejamento_fixo` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_custos_armazenagem' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_custos_armazenagem` IS NULL OR LOWER(TRIM(CAST(`cod_custos_armazenagem` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_custos_armazenagem` IS NULL OR LOWER(TRIM(CAST(`cod_custos_armazenagem` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_custos_armazenagem` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_custos_armazenagem` IS NULL OR LOWER(TRIM(CAST(`cod_custos_armazenagem` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_custos_armazenagem` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_custos_armazenagem` IS NULL OR LOWER(TRIM(CAST(`cod_custos_armazenagem` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_custos_armazenagem` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_ciclo_mrp' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_ciclo_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_ciclo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_ciclo_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_ciclo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_ciclo_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_ciclo_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_ciclo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_ciclo_mrp` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_ciclo_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_ciclo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_ciclo_mrp` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_abc' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_abc` IS NULL OR LOWER(TRIM(CAST(`cod_abc` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_abc` IS NULL OR LOWER(TRIM(CAST(`cod_abc` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_abc` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_abc` IS NULL OR LOWER(TRIM(CAST(`cod_abc` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_abc` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_abc` IS NULL OR LOWER(TRIM(CAST(`cod_abc` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_abc` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'tp_grupo_unidade_medida' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `tp_grupo_unidade_medida` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_unidade_medida` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`tp_grupo_unidade_medida` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_unidade_medida` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`tp_grupo_unidade_medida` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`tp_grupo_unidade_medida` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_unidade_medida` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`tp_grupo_unidade_medida` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `tp_grupo_unidade_medida` IS NULL OR LOWER(TRIM(CAST(`tp_grupo_unidade_medida` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`tp_grupo_unidade_medida` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_status_material_centro' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_status_material_centro` IS NULL OR LOWER(TRIM(CAST(`cod_status_material_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_status_material_centro` IS NULL OR LOWER(TRIM(CAST(`cod_status_material_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_status_material_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_status_material_centro` IS NULL OR LOWER(TRIM(CAST(`cod_status_material_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_status_material_centro` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_status_material_centro` IS NULL OR LOWER(TRIM(CAST(`cod_status_material_centro` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_status_material_centro` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_perfil_arredondamento' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `cod_perfil_arredondamento` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`cod_perfil_arredondamento` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`cod_perfil_arredondamento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`cod_perfil_arredondamento` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`cod_perfil_arredondamento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `cod_perfil_arredondamento` IS NULL OR LOWER(TRIM(CAST(`cod_perfil_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`cod_perfil_arredondamento` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'pct_refugo' AS coluna, 'decimal(5,2)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `pct_refugo` IS NULL OR LOWER(TRIM(CAST(`pct_refugo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`pct_refugo` IS NULL OR LOWER(TRIM(CAST(`pct_refugo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`pct_refugo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`pct_refugo` IS NULL OR LOWER(TRIM(CAST(`pct_refugo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`pct_refugo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `pct_refugo` IS NULL OR LOWER(TRIM(CAST(`pct_refugo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`pct_refugo` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_fixo_lote' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_tamanho_fixo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_fixo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_tamanho_fixo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_fixo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_tamanho_fixo_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_tamanho_fixo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_fixo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_tamanho_fixo_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_tamanho_fixo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_fixo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_tamanho_fixo_lote` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_maximo_lote' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_tamanho_maximo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_maximo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_tamanho_maximo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_maximo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_tamanho_maximo_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_tamanho_maximo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_maximo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_tamanho_maximo_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_tamanho_maximo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_maximo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_tamanho_maximo_lote` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_minimo_lote' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_tamanho_minimo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_minimo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_tamanho_minimo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_minimo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_tamanho_minimo_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_tamanho_minimo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_minimo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_tamanho_minimo_lote` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_tamanho_minimo_lote` IS NULL OR LOWER(TRIM(CAST(`qt_tamanho_minimo_lote` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_tamanho_minimo_lote` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_valor_arredondamento' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_valor_arredondamento` IS NULL OR LOWER(TRIM(CAST(`qt_valor_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_valor_arredondamento` IS NULL OR LOWER(TRIM(CAST(`qt_valor_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_valor_arredondamento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_valor_arredondamento` IS NULL OR LOWER(TRIM(CAST(`qt_valor_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_valor_arredondamento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_valor_arredondamento` IS NULL OR LOWER(TRIM(CAST(`qt_valor_arredondamento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_valor_arredondamento` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'vl_custos_fixos_por_pedido' AS coluna, 'decimal(13,2)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `vl_custos_fixos_por_pedido` IS NULL OR LOWER(TRIM(CAST(`vl_custos_fixos_por_pedido` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`vl_custos_fixos_por_pedido` IS NULL OR LOWER(TRIM(CAST(`vl_custos_fixos_por_pedido` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`vl_custos_fixos_por_pedido` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`vl_custos_fixos_por_pedido` IS NULL OR LOWER(TRIM(CAST(`vl_custos_fixos_por_pedido` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`vl_custos_fixos_por_pedido` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `vl_custos_fixos_por_pedido` IS NULL OR LOWER(TRIM(CAST(`vl_custos_fixos_por_pedido` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`vl_custos_fixos_por_pedido` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_estoque_maximo' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_estoque_maximo` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_maximo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_estoque_maximo` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_maximo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_estoque_maximo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_estoque_maximo` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_maximo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_estoque_maximo` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_estoque_maximo` IS NULL OR LOWER(TRIM(CAST(`qt_estoque_maximo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_estoque_maximo` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_ponto_reabastecimento' AS coluna, 'decimal(18,3)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_ponto_reabastecimento` IS NULL OR LOWER(TRIM(CAST(`qt_ponto_reabastecimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_ponto_reabastecimento` IS NULL OR LOWER(TRIM(CAST(`qt_ponto_reabastecimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_ponto_reabastecimento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_ponto_reabastecimento` IS NULL OR LOWER(TRIM(CAST(`qt_ponto_reabastecimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_ponto_reabastecimento` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_ponto_reabastecimento` IS NULL OR LOWER(TRIM(CAST(`qt_ponto_reabastecimento` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_ponto_reabastecimento` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'dt_inicio_validade_status' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `dt_inicio_validade_status` IS NULL OR LOWER(TRIM(CAST(`dt_inicio_validade_status` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`dt_inicio_validade_status` IS NULL OR LOWER(TRIM(CAST(`dt_inicio_validade_status` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`dt_inicio_validade_status` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`dt_inicio_validade_status` IS NULL OR LOWER(TRIM(CAST(`dt_inicio_validade_status` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`dt_inicio_validade_status` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `dt_inicio_validade_status` IS NULL OR LOWER(TRIM(CAST(`dt_inicio_validade_status` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`dt_inicio_validade_status` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tempo_ciclo_trabalho' AS coluna, 'decimal(5,0)' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `qt_tempo_ciclo_trabalho` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_ciclo_trabalho` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`qt_tempo_ciclo_trabalho` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_ciclo_trabalho` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`qt_tempo_ciclo_trabalho` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`qt_tempo_ciclo_trabalho` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_ciclo_trabalho` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`qt_tempo_ciclo_trabalho` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `qt_tempo_ciclo_trabalho` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_ciclo_trabalho` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`qt_tempo_ciclo_trabalho` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, 'string' AS tipo,
COUNT(*) AS total,
SUM(CASE WHEN `ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos,
SUM(CASE WHEN NOT (`ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND TRIM(CAST(`ind_marcado_exclusao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000') THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN NOT (`ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) AND NOT (TRIM(CAST(`ind_marcado_exclusao` AS STRING)) IN ('0', '0.0', '0.00', '0.000', '000', '0000')) THEN 1 ELSE 0 END) AS uteis,
ROUND(100.0 * SUM(CASE WHEN `ind_marcado_exclusao` IS NULL OR LOWER(TRIM(CAST(`ind_marcado_exclusao` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) / NULLIF(COUNT(*),0), 4) AS pct_nulos,
COUNT(DISTINCT CAST(`ind_marcado_exclusao` AS STRING)) AS distintos
FROM tmp_zglmm159_m1_centro_4058
ORDER BY pct_nulos DESC, coluna;


## 6. Colunas documentadas como sem fonte ou stub


In [0]:
SELECT
  SUM(CASE WHEN NOT (`cod_custos_armazenagem` IS NULL OR LOWER(TRIM(CAST(`cod_custos_armazenagem` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `cod_custos_armazenagem_preenchidos`,
  SUM(CASE WHEN NOT (`cod_ciclo_mrp` IS NULL OR LOWER(TRIM(CAST(`cod_ciclo_mrp` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `cod_ciclo_mrp_preenchidos`,
  SUM(CASE WHEN NOT (`pct_refugo` IS NULL OR LOWER(TRIM(CAST(`pct_refugo` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `pct_refugo_preenchidos`,
  SUM(CASE WHEN NOT (`vl_custos_fixos_por_pedido` IS NULL OR LOWER(TRIM(CAST(`vl_custos_fixos_por_pedido` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `vl_custos_fixos_por_pedido_preenchidos`,
  SUM(CASE WHEN NOT (`qt_tempo_ciclo_trabalho` IS NULL OR LOWER(TRIM(CAST(`qt_tempo_ciclo_trabalho` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-')) THEN 1 ELSE 0 END) AS `qt_tempo_ciclo_trabalho_preenchidos`
FROM tmp_zglmm159_m1_centro_4058;


## 7. Cardinalidade e valores dominantes


In [0]:
SELECT 'cod_material' AS coluna, COUNT(DISTINCT CAST(`cod_material` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_deposito' AS coluna, COUNT(DISTINCT CAST(`cod_deposito` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_centro' AS coluna, COUNT(DISTINCT CAST(`cod_centro` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_empresa' AS coluna, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'tp_grupo_mrp' AS coluna, COUNT(DISTINCT CAST(`tp_grupo_mrp` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_regra_calculo_tamanho_lote' AS coluna, COUNT(DISTINCT CAST(`cod_regra_calculo_tamanho_lote` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'tp_mrp' AS coluna, COUNT(DISTINCT CAST(`tp_mrp` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_planejador_mrp' AS coluna, COUNT(DISTINCT CAST(`cod_planejador_mrp` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_horizonte_planejamento_fixo' AS coluna, COUNT(DISTINCT CAST(`cod_horizonte_planejamento_fixo` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_custos_armazenagem' AS coluna, COUNT(DISTINCT CAST(`cod_custos_armazenagem` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_ciclo_mrp' AS coluna, COUNT(DISTINCT CAST(`cod_ciclo_mrp` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_abc' AS coluna, COUNT(DISTINCT CAST(`cod_abc` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'tp_grupo_unidade_medida' AS coluna, COUNT(DISTINCT CAST(`tp_grupo_unidade_medida` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_status_material_centro' AS coluna, COUNT(DISTINCT CAST(`cod_status_material_centro` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_perfil_arredondamento' AS coluna, COUNT(DISTINCT CAST(`cod_perfil_arredondamento` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'pct_refugo' AS coluna, COUNT(DISTINCT CAST(`pct_refugo` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_fixo_lote' AS coluna, COUNT(DISTINCT CAST(`qt_tamanho_fixo_lote` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_maximo_lote' AS coluna, COUNT(DISTINCT CAST(`qt_tamanho_maximo_lote` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_minimo_lote' AS coluna, COUNT(DISTINCT CAST(`qt_tamanho_minimo_lote` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_valor_arredondamento' AS coluna, COUNT(DISTINCT CAST(`qt_valor_arredondamento` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'vl_custos_fixos_por_pedido' AS coluna, COUNT(DISTINCT CAST(`vl_custos_fixos_por_pedido` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_estoque_maximo' AS coluna, COUNT(DISTINCT CAST(`qt_estoque_maximo` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_ponto_reabastecimento' AS coluna, COUNT(DISTINCT CAST(`qt_ponto_reabastecimento` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'dt_inicio_validade_status' AS coluna, COUNT(DISTINCT CAST(`dt_inicio_validade_status` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tempo_ciclo_trabalho' AS coluna, COUNT(DISTINCT CAST(`qt_tempo_ciclo_trabalho` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'ind_marcado_exclusao' AS coluna, COUNT(DISTINCT CAST(`ind_marcado_exclusao` AS STRING)) AS distintos FROM tmp_zglmm159_m1_centro_4058
ORDER BY distintos ASC, coluna;


In [0]:
-- Domínio: cod_material
SELECT CAST(`cod_material` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_material` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_deposito
SELECT CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_deposito` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_centro
SELECT CAST(`cod_centro` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_centro` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_empresa
SELECT CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_empresa` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: tp_grupo_mrp
SELECT CAST(`tp_grupo_mrp` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`tp_grupo_mrp` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_regra_calculo_tamanho_lote
SELECT CAST(`cod_regra_calculo_tamanho_lote` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_regra_calculo_tamanho_lote` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: tp_mrp
SELECT CAST(`tp_mrp` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`tp_mrp` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_planejador_mrp
SELECT CAST(`cod_planejador_mrp` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_planejador_mrp` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_horizonte_planejamento_fixo
SELECT CAST(`cod_horizonte_planejamento_fixo` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_horizonte_planejamento_fixo` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_custos_armazenagem
SELECT CAST(`cod_custos_armazenagem` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_custos_armazenagem` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_ciclo_mrp
SELECT CAST(`cod_ciclo_mrp` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_ciclo_mrp` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_abc
SELECT CAST(`cod_abc` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_abc` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: tp_grupo_unidade_medida
SELECT CAST(`tp_grupo_unidade_medida` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`tp_grupo_unidade_medida` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_status_material_centro
SELECT CAST(`cod_status_material_centro` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_status_material_centro` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: cod_perfil_arredondamento
SELECT CAST(`cod_perfil_arredondamento` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`cod_perfil_arredondamento` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: dt_inicio_validade_status
SELECT CAST(`dt_inicio_validade_status` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`dt_inicio_validade_status` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


In [0]:
-- Domínio: ind_marcado_exclusao
SELECT CAST(`ind_marcado_exclusao` AS STRING) AS valor, COUNT(*) AS qtd,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct
FROM tmp_zglmm159_m1_centro_4058
GROUP BY CAST(`ind_marcado_exclusao` AS STRING)
ORDER BY qtd DESC
LIMIT 20;


## 8. Perfil numérico e extremos


In [0]:
SELECT 'pct_refugo' AS coluna,
MIN(`pct_refugo`) AS minimo,
MAX(`pct_refugo`) AS maximo,
AVG(`pct_refugo`) AS media,
SUM(`pct_refugo`) AS soma,
SUM(CASE WHEN `pct_refugo` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `pct_refugo` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `pct_refugo` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_fixo_lote' AS coluna,
MIN(`qt_tamanho_fixo_lote`) AS minimo,
MAX(`qt_tamanho_fixo_lote`) AS maximo,
AVG(`qt_tamanho_fixo_lote`) AS media,
SUM(`qt_tamanho_fixo_lote`) AS soma,
SUM(CASE WHEN `qt_tamanho_fixo_lote` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_tamanho_fixo_lote` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_tamanho_fixo_lote` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_maximo_lote' AS coluna,
MIN(`qt_tamanho_maximo_lote`) AS minimo,
MAX(`qt_tamanho_maximo_lote`) AS maximo,
AVG(`qt_tamanho_maximo_lote`) AS media,
SUM(`qt_tamanho_maximo_lote`) AS soma,
SUM(CASE WHEN `qt_tamanho_maximo_lote` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_tamanho_maximo_lote` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_tamanho_maximo_lote` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tamanho_minimo_lote' AS coluna,
MIN(`qt_tamanho_minimo_lote`) AS minimo,
MAX(`qt_tamanho_minimo_lote`) AS maximo,
AVG(`qt_tamanho_minimo_lote`) AS media,
SUM(`qt_tamanho_minimo_lote`) AS soma,
SUM(CASE WHEN `qt_tamanho_minimo_lote` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_tamanho_minimo_lote` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_tamanho_minimo_lote` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_valor_arredondamento' AS coluna,
MIN(`qt_valor_arredondamento`) AS minimo,
MAX(`qt_valor_arredondamento`) AS maximo,
AVG(`qt_valor_arredondamento`) AS media,
SUM(`qt_valor_arredondamento`) AS soma,
SUM(CASE WHEN `qt_valor_arredondamento` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_valor_arredondamento` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_valor_arredondamento` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'vl_custos_fixos_por_pedido' AS coluna,
MIN(`vl_custos_fixos_por_pedido`) AS minimo,
MAX(`vl_custos_fixos_por_pedido`) AS maximo,
AVG(`vl_custos_fixos_por_pedido`) AS media,
SUM(`vl_custos_fixos_por_pedido`) AS soma,
SUM(CASE WHEN `vl_custos_fixos_por_pedido` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `vl_custos_fixos_por_pedido` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `vl_custos_fixos_por_pedido` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_estoque_maximo' AS coluna,
MIN(`qt_estoque_maximo`) AS minimo,
MAX(`qt_estoque_maximo`) AS maximo,
AVG(`qt_estoque_maximo`) AS media,
SUM(`qt_estoque_maximo`) AS soma,
SUM(CASE WHEN `qt_estoque_maximo` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_estoque_maximo` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_estoque_maximo` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_ponto_reabastecimento' AS coluna,
MIN(`qt_ponto_reabastecimento`) AS minimo,
MAX(`qt_ponto_reabastecimento`) AS maximo,
AVG(`qt_ponto_reabastecimento`) AS media,
SUM(`qt_ponto_reabastecimento`) AS soma,
SUM(CASE WHEN `qt_ponto_reabastecimento` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_ponto_reabastecimento` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_ponto_reabastecimento` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'qt_tempo_ciclo_trabalho' AS coluna,
MIN(`qt_tempo_ciclo_trabalho`) AS minimo,
MAX(`qt_tempo_ciclo_trabalho`) AS maximo,
AVG(`qt_tempo_ciclo_trabalho`) AS media,
SUM(`qt_tempo_ciclo_trabalho`) AS soma,
SUM(CASE WHEN `qt_tempo_ciclo_trabalho` < 0 THEN 1 ELSE 0 END) AS negativos,
SUM(CASE WHEN `qt_tempo_ciclo_trabalho` = 0 THEN 1 ELSE 0 END) AS zeros,
SUM(CASE WHEN `qt_tempo_ciclo_trabalho` > 0 THEN 1 ELSE 0 END) AS positivos
FROM tmp_zglmm159_m1_centro_4058;


## 9. Datas, formatos e faixas


In [0]:
SELECT 'dt_inicio_validade_status' AS coluna,
MIN(CAST(`dt_inicio_validade_status` AS STRING)) AS minimo_bruto,
MAX(CAST(`dt_inicio_validade_status` AS STRING)) AS maximo_bruto,
COUNT(DISTINCT CAST(`dt_inicio_validade_status` AS STRING)) AS distintos,
SUM(CASE WHEN CAST(`dt_inicio_validade_status` AS STRING) RLIKE '^\d{8}$' THEN 1 ELSE 0 END) AS formato_aaaammdd,
SUM(CASE WHEN `dt_inicio_validade_status` IS NULL OR LOWER(TRIM(CAST(`dt_inicio_validade_status` AS STRING))) IN ('', 'null', 'none', 'nan', '#', '-') THEN 1 ELSE 0 END) AS nulos
FROM tmp_zglmm159_m1_centro_4058;


## 10. Zeros à esquerda e formato das chaves


In [0]:
SELECT 'cod_material' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_material` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_material` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_material` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_material` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_centro' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_centro` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_centro` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_centro` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_centro` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_empresa' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_empresa` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_empresa` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_empresa` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_empresa` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM tmp_zglmm159_m1_centro_4058
UNION ALL
SELECT 'cod_deposito' AS coluna,
MIN(LENGTH(TRIM(CAST(`cod_deposito` AS STRING)))) AS tamanho_min,
MAX(LENGTH(TRIM(CAST(`cod_deposito` AS STRING)))) AS tamanho_max,
SUM(CASE WHEN TRIM(CAST(`cod_deposito` AS STRING)) RLIKE '^0[0-9]+' THEN 1 ELSE 0 END) AS com_zero_esquerda,
COUNT(DISTINCT TRIM(CAST(`cod_deposito` AS STRING))) AS distintos_brutos,
COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(`cod_deposito` AS STRING)), '^0+', '')) AS distintos_sem_zero
FROM tmp_zglmm159_m1_centro_4058;


## 11. Linhas integralmente duplicadas


In [0]:
SELECT
  COUNT(*) AS linhas,
  COUNT(DISTINCT SHA2(CONCAT_WS('§', COALESCE(CAST(`cod_material` AS STRING), '<NULL>'), COALESCE(CAST(`cod_deposito` AS STRING), '<NULL>'), COALESCE(CAST(`cod_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_empresa` AS STRING), '<NULL>'), COALESCE(CAST(`tp_grupo_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_regra_calculo_tamanho_lote` AS STRING), '<NULL>'), COALESCE(CAST(`tp_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_planejador_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_horizonte_planejamento_fixo` AS STRING), '<NULL>'), COALESCE(CAST(`cod_custos_armazenagem` AS STRING), '<NULL>'), COALESCE(CAST(`cod_ciclo_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_abc` AS STRING), '<NULL>'), COALESCE(CAST(`tp_grupo_unidade_medida` AS STRING), '<NULL>'), COALESCE(CAST(`cod_status_material_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_perfil_arredondamento` AS STRING), '<NULL>'), COALESCE(CAST(`pct_refugo` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tamanho_fixo_lote` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tamanho_maximo_lote` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tamanho_minimo_lote` AS STRING), '<NULL>'), COALESCE(CAST(`qt_valor_arredondamento` AS STRING), '<NULL>'), COALESCE(CAST(`vl_custos_fixos_por_pedido` AS STRING), '<NULL>'), COALESCE(CAST(`qt_estoque_maximo` AS STRING), '<NULL>'), COALESCE(CAST(`qt_ponto_reabastecimento` AS STRING), '<NULL>'), COALESCE(CAST(`dt_inicio_validade_status` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tempo_ciclo_trabalho` AS STRING), '<NULL>'), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '<NULL>')), 256)) AS linhas_distintas,
  COUNT(*) - COUNT(DISTINCT SHA2(CONCAT_WS('§', COALESCE(CAST(`cod_material` AS STRING), '<NULL>'), COALESCE(CAST(`cod_deposito` AS STRING), '<NULL>'), COALESCE(CAST(`cod_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_empresa` AS STRING), '<NULL>'), COALESCE(CAST(`tp_grupo_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_regra_calculo_tamanho_lote` AS STRING), '<NULL>'), COALESCE(CAST(`tp_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_planejador_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_horizonte_planejamento_fixo` AS STRING), '<NULL>'), COALESCE(CAST(`cod_custos_armazenagem` AS STRING), '<NULL>'), COALESCE(CAST(`cod_ciclo_mrp` AS STRING), '<NULL>'), COALESCE(CAST(`cod_abc` AS STRING), '<NULL>'), COALESCE(CAST(`tp_grupo_unidade_medida` AS STRING), '<NULL>'), COALESCE(CAST(`cod_status_material_centro` AS STRING), '<NULL>'), COALESCE(CAST(`cod_perfil_arredondamento` AS STRING), '<NULL>'), COALESCE(CAST(`pct_refugo` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tamanho_fixo_lote` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tamanho_maximo_lote` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tamanho_minimo_lote` AS STRING), '<NULL>'), COALESCE(CAST(`qt_valor_arredondamento` AS STRING), '<NULL>'), COALESCE(CAST(`vl_custos_fixos_por_pedido` AS STRING), '<NULL>'), COALESCE(CAST(`qt_estoque_maximo` AS STRING), '<NULL>'), COALESCE(CAST(`qt_ponto_reabastecimento` AS STRING), '<NULL>'), COALESCE(CAST(`dt_inicio_validade_status` AS STRING), '<NULL>'), COALESCE(CAST(`qt_tempo_ciclo_trabalho` AS STRING), '<NULL>'), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '<NULL>')), 256)) AS duplicatas_integrais
FROM tmp_zglmm159_m1_centro_4058;


## 12. Amostra ampliada — validar formatos reais


In [0]:
SELECT * FROM tmp_zglmm159_m1_centro_4058 ORDER BY cod_centro, cod_material LIMIT 100;


## 13. Resumo automático de cenário e chave


In [0]:
WITH base AS (
  SELECT COUNT(*) AS linhas,
         COUNT(DISTINCT CONCAT(COALESCE(TRIM(CAST(`cod_material` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_centro` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_deposito` AS STRING)), '<NULL>'), '|', COALESCE(TRIM(CAST(`cod_empresa` AS STRING)), '<NULL>'))) AS chaves
  FROM tmp_zglmm159_m1_centro_4058
)
SELECT 'CENARIO' AS bloco, 'tabela' AS item, 'tmp_zglmm159_m1_centro_4058' AS valor
UNION ALL SELECT 'CENARIO', 'colunas', '26'
UNION ALL SELECT 'CENARIO', 'linhas', CAST(linhas AS STRING) FROM base
UNION ALL SELECT 'CHAVE CANDIDATA', 'campos', 'cod_material + cod_centro + cod_deposito + cod_empresa'
UNION ALL SELECT 'CHAVE CANDIDATA', 'distintos', CAST(chaves AS STRING) FROM base
UNION ALL SELECT 'CHAVE CANDIDATA', 'veredito', CASE WHEN linhas = chaves THEN 'CHAVE UNICA' ELSE 'NAO UNICA - INVESTIGAR DIMENSAO ADICIONAL' END FROM base;


---
## Próximo passo

1. Revisar a granularidade real e a chave candidata.
2. Confirmar se as colunas documentadas como sem fonte permanecem realmente nulas.
3. Escolher um recorte reproduzível para extração SAP, preservando todas as visões C1, M1 e M2.
4. Não avançar para o de/para antes de validar o diagnóstico desta análise exploratória.


## Controle específico do cenário anterior

A validação anterior do centro 4058 tratou M1 como 1 linha por material + depósito. Este notebook mede explicitamente materiais com múltiplos depósitos e confirma se a nova view preserva todas as linhas dessa granularidade.


In [0]:
SELECT
  cod_material,
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_deposito) AS depositos,
  CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(cod_deposito))) AS lista_depositos
FROM tmp_zglmm159_m1_centro_4058
GROUP BY cod_material
HAVING COUNT(DISTINCT cod_deposito) > 1
ORDER BY depositos DESC, linhas DESC, cod_material;


In [0]:
SELECT *
FROM dev_procurement.corp_curated.vw_zglmm159_m1
WHERE TRIM(CAST(cod_centro AS STRING)) = '4058';

## Portão do cenário 4058

Antes de extrair novamente o SAP, confirme:
- a volumetria da nova visão para o centro;
- a chave e a granularidade real;
- a preservação dos múltiplos depósitos na M1;
- o preenchimento real das colunas documentadas como sem fonte;
- a existência de valores ou dimensões inesperadas.
